<a href="https://colab.research.google.com/github/AbhinandAG/AI-ML-API_Obesity/blob/main/ObesityTrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from flask import Flask, request, jsonify, render_template
import pandas as pd
import joblib

app = Flask("ObesityPredictor")

# Load model and scaler
model = joblib.load("obesity_trnd.pkl")
x_scaler = joblib.load("obesityXscaler.pkl")

# Load encoders
Gender_encoder = joblib.load("Gender_encoder.pkl")
CALC_encoder = joblib.load("CALC_encoder.pkl")
FAVC_encoder = joblib.load("FAVC_encoder.pkl")
SCC_encoder = joblib.load("SCC_encoder.pkl")
SMOKE_encoder = joblib.load("SMOKE_encoder.pkl")
family_encoder = joblib.load(
    "family_history_with_overweight_encoder.pkl"
)
CAEC_encoder = joblib.load("CAEC_encoder.pkl")
MTRANS_encoder = joblib.load("MTRANS_encoder.pkl")


LABELS = {
    0: "Insufficient_Weight",
    1: "Normal_Weight",
    2: "Obesity_Type_I",
    3: "Obesity_Type_II",
    4: "Obesity_Type_III",
    5: "Overweight_Level_I",
    6: "Overweight_Level_II"
}


@app.route("/")
def home():
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    try:

        Age = float(data["Age"])
        Gender = Gender_encoder.transform([data["Gender"]])[0]
        Height = float(data["Height"])
        Weight = float(data["Weight"])

        CALC = CALC_encoder.transform([data["CALC"]])[0]
        FAVC = FAVC_encoder.transform([data["FAVC"]])[0]

        FCVC = float(data["FCVC"])
        NCP = float(data["NCP"])

        SCC = SCC_encoder.transform([data["SCC"]])[0]
        SMOKE = SMOKE_encoder.transform([data["SMOKE"]])[0]

        CH2O = float(data["CH2O"])

        family = family_encoder.transform(
            [data["family_history_with_overweight"]]
        )[0]

        FAF = float(data["FAF"])
        TUE = float(data["TUE"])

        CAEC = CAEC_encoder.transform([data["CAEC"]])[0]
        MTRANS = MTRANS_encoder.transform([data["MTRANS"]])[0]

        user_input = pd.DataFrame([[
            Age,
            Gender,
            Height,
            Weight,
            CALC,
            FAVC,
            FCVC,
            NCP,
            SCC,
            SMOKE,
            CH2O,
            family,
            FAF,
            TUE,
            CAEC,
            MTRANS
        ]])

        scaled = x_scaler.transform(user_input)

        prediction = model.predict(scaled)

        pred_class = int(prediction[0])

        result = LABELS.get(pred_class, "Unknown")

        return jsonify({
            "success": True,
            "prediction_code": pred_class,
            "prediction": result
        })

    except Exception as e:
        return jsonify({
            "success": False,
            "error": str(e)
        })


if __name__ == "__main__":
    app.run(debug=True)

<!DOCTYPE html>
<html>
<head>
    <title>Obesity Prediction AI</title>

    <link rel="stylesheet" href="/static/style.css">
</head>
<body>

<div class="container">

    <h1>Obesity Prediction</h1>

    <form id="predictForm">

        <input type="number" step="0.1" placeholder="Age" id="Age">

        <select id="Gender">
            <option>Male</option>
            <option>Female</option>
        </select>

        <input type="number" step="0.01" placeholder="Height" id="Height">
        <input type="number" step="0.01" placeholder="Weight" id="Weight">

        <select id="CALC">
            <option>no</option>
            <option>Sometimes</option>
            <option>Frequently</option>
            <option>Always</option>
        </select>

        <select id="FAVC">
            <option>yes</option>
            <option>no</option>
        </select>

        <input type="number" step="0.1" placeholder="FCVC" id="FCVC">
        <input type="number" step="0.1" placeholder="NCP" id="NCP">

        <select id="SCC">
            <option>yes</option>
            <option>no</option>
        </select>

        <select id="SMOKE">
            <option>yes</option>
            <option>no</option>
        </select>

        <input type="number" step="0.1" placeholder="CH2O" id="CH2O">

        <select id="family_history_with_overweight">
            <option>yes</option>
            <option>no</option>
        </select>

        <input type="number" step="0.1" placeholder="FAF" id="FAF">
        <input type="number" step="0.1" placeholder="TUE" id="TUE">

        <select id="CAEC">
            <option>no</option>
            <option>Sometimes</option>
            <option>Frequently</option>
            <option>Always</option>
        </select>

        <select id="MTRANS">
            <option>Public_Transportation</option>
            <option>Walking</option>
            <option>Automobile</option>
            <option>Motorbike</option>
            <option>Bike</option>
        </select>

        <button type="submit">
            Predict
        </button>

    </form>

    <div id="result"></div>

</div>

<script src="/static/app.js"></script>

</body>
</html>